# Learning Engine Simulation Campaign — all stimulus types

This notebook builds a `LearningEngineCircuitSimulationScanConfig` campaign with every
stimulus type that the Learning Engine simulator accepts (the
`LearningEngineCircuitStimulusUnion`).

**Absolute current-clamp stimuli**

- `ConstantCurrentClampSomaticStimulus`
- `LinearCurrentClampSomaticStimulus`
- `MultiPulseCurrentClampSomaticStimulus`
- `SinusoidalCurrentClampSomaticStimulus`

**Relative current-clamp stimuli (percentage of each cell's threshold current)**

- `RelativeConstantCurrentClampSomaticStimulus`
- `RelativeLinearCurrentClampSomaticStimulus`
- `SubthresholdCurrentClampSomaticStimulus`

The campaign is generated against an entitycore-staged circuit on the staging
deployment. Each grid coordinate writes a `simulation_config.json` with
`target_simulator: "LearningEngine"` ready to hand off to the Learning Engine runner.

In [ ]:
import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token

from obi_one.scientific.tasks.generate_simulations.config.learning_engine.le_circuit import (
    LearningEngineCircuitSimulationScanConfig,
)

## Stage the circuit from entitycore

Authenticate against the staging deployment and point `obi.CircuitFromID` at the
target circuit. The circuit is staged to `coordinate_output_root/sonata_circuit/`
on each grid coordinate when `grid_scan.execute(db_client=db_client)` runs.

In [ ]:
virtual_lab_id = "84258ff5-114f-4865-9a2d-258575c23909"
project_id = "b3d4ac06-f636-465c-a6e6-72dd098509a1"
circuit_id = "32f54457-6196-491d-8f0c-58afb3759dae"

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)

circuit = obi.CircuitFromID(id_str=circuit_id)
print(f"Using entitycore circuit {circuit_id}")

## Build the campaign skeleton

`empty_config()` returns a `LearningEngineCircuitSimulationScanConfig` with all
blocks unset. We fill it in by adding timestamps and neuron sets that the stimuli
will reference.

In [ ]:
sim_duration = 1000.0

sim_conf = LearningEngineCircuitSimulationScanConfig.empty_config()

# Campaign metadata
sim_conf.set(
    obi.Info(
        campaign_name="LE — all stimulus types",
        campaign_description=(
            "Demonstrates building a Learning Engine simulation campaign with every "
            "stimulus type supported by LearningEngineCircuitStimulusUnion."
        ),
    ),
    name="info",
)

# A single timestamp block reused by every stimulus
timestamps = obi.RegularTimestamps(start_time=0.0, number_of_repetitions=2, interval=500.0)
sim_conf.add(timestamps, name="StimTimestamps")

# Biophysical target neurons
sim_neurons = obi.AllNeurons()
sim_conf.add(sim_neurons, name="AllNeurons")

## Current-clamp stimuli

The seven somatic current-clamp stimuli supported by the Learning Engine — four
absolute-amplitude and three relative-to-threshold. We make
`ConstantCurrentClampSomaticStimulus.amplitude` a list so the campaign expands
into multiple grid coordinates.

In [ ]:
# Absolute current-clamp stimuli
sim_conf.add(
    obi.ConstantCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        amplitude=[0.1, 0.3],  # scan parameter → 2 coordinates
    ),
    name="ConstantCurrent",
)

sim_conf.add(
    obi.LinearCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        amplitude_start=0.0,
        amplitude_end=0.3,
    ),
    name="LinearCurrent",
)

sim_conf.add(
    obi.MultiPulseCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=300.0,
        amplitude=0.2,
        width=5.0,
        frequency=10.0,
    ),
    name="MultiPulseCurrent",
)

sim_conf.add(
    obi.SinusoidalCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=300.0,
        maximum_amplitude=0.15,
        frequency=8.0,
        dt=0.025,
    ),
    name="SinusoidalCurrent",
)

# Relative current-clamp stimuli (percentage of each cell's threshold current)
sim_conf.add(
    obi.RelativeConstantCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        percentage_of_threshold_current=20.0,
    ),
    name="RelativeConstantCurrent",
)

sim_conf.add(
    obi.RelativeLinearCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        percentage_of_threshold_current_start=0.0,
        percentage_of_threshold_current_end=50.0,
    ),
    name="RelativeLinearCurrent",
)

sim_conf.add(
    obi.SubthresholdCurrentClampSomaticStimulus(
        timestamps=timestamps.ref,
        neuron_set=sim_neurons.ref,
        duration=100.0,
        percentage_below_threshold=20.0,
    ),
    name="SubthresholdCurrent",
)

## Initialize and validate

The Initialize block pins the campaign to a circuit and a simulation length. The
`random_seed` field is hidden in the UI but still scannable from Python.

In [ ]:
sim_conf.set(
    LearningEngineCircuitSimulationScanConfig.Initialize(
        circuit=circuit,
        node_set=sim_neurons.ref,
        simulation_length=sim_duration,
        v_init=-80.0,
        random_seed=1,
    ),
    name="initialize",
)

validated_sim_conf = sim_conf.validated_config()
print(validated_sim_conf)

## Generate the grid scan

Only `ConstantCurrent.amplitude` is a list, so the grid expands into 2 coordinates.
Each coordinate writes a `simulation_config.json` and stages the SONATA circuit.

In [ ]:
output_root = "../../../../obi-output/learning_engine_simulations/grid_scan"

grid_scan = obi.GridScanGenerationTask(
    form=validated_sim_conf,
    coordinate_directory_option="ZERO_INDEX",
    output_root=output_root,
)
grid_scan.multiple_value_parameters(display=True)
grid_scan.execute(db_client=db_client)
obi.run_task_for_single_configs(single_configs=grid_scan.single_configs, db_client=db_client)

## Inspect a generated `simulation_config.json`

The Learning Engine config carries `target_simulator: "LearningEngine"` and the
same SONATA `run` / `inputs` structure that the Learning Engine runner expects.

In [ ]:
import json

simulation_config_path = (
    grid_scan.single_configs[0].coordinate_output_root / "simulation_config.json"
)
print(f"simulation_config: {simulation_config_path}\n")

with simulation_config_path.open() as f:
    generated = json.load(f)

print(f"target_simulator: {generated['target_simulator']}")
print(f"run:              {generated['run']}")
print(f"conditions:       {generated['conditions']}")
print(f"inputs:           {list(generated.get('inputs', {}).keys())}")